<a href="https://colab.research.google.com/github/ompatel420/FreeCodeCamp-Machine-Learning-Project/blob/main/Neural_Network_SMS_Text_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

In [2]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

dataset = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

dataset.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
# Split dataset into training and testing data

from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    dataset,
    test_size=0.2,
    random_state=42,
    stratify=dataset["label"]
)

# Convert messages and labels into arrays

train_messages = train_data["message"].astype(str).values
train_labels = train_data["label"].values

test_messages = test_data["message"].astype(str).values
test_labels = test_data["label"].values

# Convert ham and spam into numbers
train_labels = np.array([1 if label == "spam" else 0 for label in train_labels])
test_labels = np.array([1 if label == "spam" else 0 for label in test_labels])

print("Training messages:", len(train_messages))
print("Testing messages:", len(test_messages))

Training messages: 4457
Testing messages: 1115


In [5]:
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)

In [6]:
vocab_size = 10000
max_length = 100

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(train_messages)

# Convert training messages
training_sequences = tokenizer.texts_to_sequences(
    train_messages
)

training_padded = pad_sequences(
    training_sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

# Convert testing messages
testing_sequences = tokenizer.texts_to_sequences(
    test_messages
)

testing_padded = pad_sequences(
    testing_sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print("Tokenization completed!")
print("Training shape:", training_padded.shape)
print("Testing shape:", testing_padded.shape)

Tokenization completed!
Training shape: (4457, 100)
Testing shape: (1115, 100)


In [7]:
model = Sequential([
    Embedding(
        vocab_size,
        64,
        input_length=max_length
    ),

    GlobalAveragePooling1D(),

    Dense(64, activation='relu'),

    Dropout(0.3),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    training_padded,
    train_labels,
    epochs=15,
    validation_split=0.2,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8698 - loss: 0.3875 - val_accuracy: 0.8430 - val_loss: 0.4051
Epoch 2/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.8715 - loss: 0.3552 - val_accuracy: 0.8430 - val_loss: 0.3855
Epoch 3/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8715 - loss: 0.3258 - val_accuracy: 0.8430 - val_loss: 0.3496
Epoch 4/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9167 - loss: 0.1945 - val_accuracy: 0.9596 - val_loss: 0.1210
Epoch 5/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9798 - loss: 0.0804 - val_accuracy: 0.9686 - val_loss: 0.0804
Epoch 6/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9849 - loss: 0.0555 - val_accuracy: 0.9451 - val_loss: 0.1857
Epoch 7/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9879 - loss: 0.0460 - val_accuracy: 0.9675 - val_loss: 0.1065
Epoch 8/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9899 - loss: 0.0364 - val_accu

In [9]:
loss, accuracy = model.evaluate(
    testing_padded,
    test_labels,
    verbose=2
)

print("Test Accuracy:", accuracy * 100, "%")

35/35 - 0s - 4ms/step - accuracy: 0.9785 - loss: 0.0732
Test Accuracy: 97.84753322601318 %


In [10]:
def predict_message(message):

    # Convert message into sequence
    sequence = tokenizer.texts_to_sequences([message])

    # Pad sequence
    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding='post',
        truncating='post'
    )

    # Predict spam probability
    prediction = float(model.predict(padded, verbose=0)[0][0])

    # Classify message
    label = "spam" if prediction >= 0.5 else "ham"

    return [prediction, label]

In [11]:
messages = [
    "Hey, how are you? Let's meet tomorrow.",
    "Congratulations! You won a free prize. Call now!",
    "URGENT! Claim your free cash reward now!",
    "Are you coming to college today?"
]

for message in messages:
    print("\nMessage:", message)
    print("Prediction:", predict_message(message))


Message: Hey, how are you? Let's meet tomorrow.
Prediction: [0.0024019356351345778, 'ham']

Message: Congratulations! You won a free prize. Call now!
Prediction: [0.6359037756919861, 'spam']

Message: URGENT! Claim your free cash reward now!
Prediction: [0.5090463161468506, 'spam']

Message: Are you coming to college today?
Prediction: [0.006495998241007328, 'ham']


In [12]:
# Test your model with sample messages

test_messages = [
    "Congratulations! You have won a $1000 Walmart gift card. Go to http://bit.ly/123456 to claim now.",
    "Hey, are we still meeting for dinner tonight?",
    "URGENT! You have won a 1 week FREE membership in our £100,000 Prize Jackpot!",
    "I'm going to be home soon and I don't want to talk about this stuff anymore.",
    "You have been selected to receive a $500 Amazon gift card. Claim your prize now!"
]

for message in test_messages:
    prediction = predict_message(message)
    print("Message:", message)
    print("Prediction:", prediction)
    print()

Message: Congratulations! You have won a $1000 Walmart gift card. Go to http://bit.ly/123456 to claim now.
Prediction: [0.8302789330482483, 'spam']

Message: Hey, are we still meeting for dinner tonight?
Prediction: [0.0023050373420119286, 'ham']

Message: URGENT! You have won a 1 week FREE membership in our £100,000 Prize Jackpot!
Prediction: [0.8114414215087891, 'spam']

Message: I'm going to be home soon and I don't want to talk about this stuff anymore.
Prediction: [0.001409426680766046, 'ham']

Message: You have been selected to receive a $500 Amazon gift card. Claim your prize now!
Prediction: [0.9394993185997009, 'spam']

